# AIVoice Phase 28 — Kaggle Universal Emotion PoC

Research only. No production AIVoice integration. Follow every cell in order and do not bypass the smoke-listening gate.

## 00 — Environment Check

In [ ]:
from pathlib import Path
import os, sys
POC_ROOT = Path('/kaggle/working/index_emotion_kaggle')
if not POC_ROOT.is_dir():
    raise RuntimeError('Copy the attached experiment package to /kaggle/working/index_emotion_kaggle first. See README.md.')
os.chdir(POC_ROOT)
!python scripts/verify_environment.py

## 01 — Install Pinned Dependencies

Keeps Kaggle's CUDA-enabled PyTorch; no training, Gradio, DeepSpeed, custom kernels, or Qwen text-emotion assets.

In [ ]:
%cd {POC_ROOT}
!pip install -q -r requirements-lock.txt
!git clone --filter=blob:none https://github.com/iamdinhthuan/index-tts-finetune-vietnamese.git community-source
%cd {POC_ROOT}/community-source
!git checkout --detach f63b1879d4b46aac456bd00640a4833a2ae071f4
!pip install -q --no-deps -e .
%cd {POC_ROOT}
!python -c "import torch, indextts; print('torch=', torch.__version__, 'cuda=', torch.version.cuda, 'indextts import PASS')"

## 02 — Configure Model Sources

Inspect the approved manifest. The Qwen text-emotion classifier is intentionally skipped.

In [ ]:
%cd {POC_ROOT}
!python -m json.tool assets_manifest.json | head -160
!grep -R "qwen0.6bemo4-merge" -n assets_manifest.json poc_config.json

## 03 — Download and Verify Approved Assets

Do not set approval until the environment check passed and the checkpoint/provenance decision is approved. This is the first cell that can download model files.

In [ ]:
%cd {POC_ROOT}
DOWNLOAD_APPROVED = False  # Change manually only after reviewing sources and free disk.
if not DOWNLOAD_APPROVED:
    print('STOP: asset download is intentionally blocked. Set DOWNLOAD_APPROVED=True to continue.')
else:
    !python scripts/download_assets.py --confirm-download
    !python scripts/verify_assets.py

## 04 — Verify Hashes / Revisions

Run after the approved download. No Qwen folder may be present.

In [ ]:
%cd {POC_ROOT}
!python scripts/verify_assets.py

## 05 — Load LOW-VRAM Engine

The serial smoke/matrix runners load the engine exactly once per run after all preflights. Do not instantiate another WebUI or second engine.

In [ ]:
%cd {POC_ROOT}
!python scripts/verify_assets.py
print('Preflight PASS. run_smoke.py will perform the one-engine load.')

## 06 — Upload References

Manually upload only consented files described in reference_checklist.md. Never upload AIVoice source, saved voices, databases, API keys, or unrelated recordings.

In [ ]:
%cd {POC_ROOT}
!cat reference_checklist.md
print('Required smoke files: references/speakers/speaker_A.wav, references/speakers/speaker_B.wav, references/emotions/emotion_sad.wav')

## 07 — Smoke Test

Generates only A-neutral, A-sad, and B-sad. It stops after them.

In [ ]:
%cd {POC_ROOT}
!python scripts/run_smoke.py

## 08 — Universal Emotion Transfer Listening Gate

Listen to all three smoke outputs. Check Vietnamese pronunciation, identity, sadness, naturalness, and donor leakage. Keep `smoke_test_approved` false on any serious failure.

In [ ]:
%cd {POC_ROOT}
!cat evaluation_template.md
print('Manual gate: edit poc_config.json and set smoke_test_approved=true ONLY after human listening approves.')

## 09 — Full 12-Sample Matrix

This command refuses to run until the manual smoke approval is true. It generates exactly A/B/C × neutral/happy/sad/angry, one file at a time, with resume support.

In [ ]:
%cd {POC_ROOT}
!python scripts/run_matrix.py
# Optional, never automatic, after shared-audio evaluation succeeds:
# !python scripts/run_vector_test.py

## 10 — Package Outputs

Creates a small ZIP without checkpoints, caches, or reference audio.

In [ ]:
%cd {POC_ROOT}
!python scripts/package_results.py
!unzip -l aivoice_phase28_poc.zip